In [1]:
import numpy as np

eps = 1e-10
kmax = 1000

In [ ]:
def find_max_offdiag(A):
    n = A.shape[0]
    p, q = 1, 0
    max_val = abs(A[p, q])
    for i in range(1, n):
        for j in range(i):
            val = abs(A[i, j])
            if val > max_val:
                max_val = val
                p, q = i, j
    return p, q


def compute_cs(A, p, q):
    # alpha = (a_pp - a_qq) / (2 * a_pq)
    alpha = (A[p, p] - A[q, q]) / (2.0 * A[p, q])

    # t = -alpha + semn(alpha) * sqrt(alpha^2 + 1)
    if alpha >= 0:
        t = -alpha + np.sqrt(alpha**2 + 1)
    else:
        t = -alpha - np.sqrt(alpha**2 + 1)

    # c = 1 / sqrt(1 + t^2),  s = t / sqrt(1 + t^2)
    c = 1.0 / np.sqrt(1.0 + t**2)
    s = t / np.sqrt(1.0 + t**2)
    return c, s, t


def jacobi(A, eps=eps, kmax=kmax):
    if not np.allclose(A, A.T):
        raise ValueError("Jacobi: A trebuie sa fie simetrica (A = A^T).")

    n = A.shape[0]
    A = A.copy().astype(float)
    # U = I_n
    U = np.eye(n)
    k = 0
    p, q = find_max_offdiag(A)

    while abs(A[p, q]) > eps and k <= kmax:
        c, s, t = compute_cs(A, p, q)
        apq = A[p, q]

        for j in range(n):
            if j != p and j != q:
                apj = A[p, j]
                aqj = A[q, j]
                # a_pj = c*a_pj + s*a_qj,
                A[j, p] = A[p, j] = c * apj + s * aqj
                # a_qj = -s*a_pj + c*a_qj,
                A[j, q] = A[q, j] = -s * apj + c * aqj

        # a_pp = a_pp + t*a_pq
        A[p, p] = A[p, p] + t * apq
        # a_qq = a_qq - t*a_pq
        A[q, q] = A[q, q] - t * apq
        # a_pq = a_qp = 0
        A[p, q] = A[q, p] = 0.0

        for i in range(n):
            uip = U[i, p]
            uiq = U[i, q]
            # u_ip = c*u_ip + s*u_iq
            U[i, p] = c * uip + s * uiq
            # u_iq = -s*u_ip_vechi + c*u_iq
            U[i, q] = -s * uip + c * uiq

        # recalculeaza p, q
        p, q = find_max_offdiag(A)
        k += 1

    eigenvalues = np.diag(A).copy()
    return eigenvalues, U

In [42]:
matrices = [
    np.array([[0, 0, 1], [0, 0, 1], [1, 1, 1]], dtype=float),
    np.array([[1, 1, 2], [1, 1, 2], [2, 2, 2]], dtype=float),
    np.array([[1, 0, 1, 0], [0, 1, 0, 1], [1, 0, 1, 0], [0, 1, 0, 1]], dtype=float),
    np.array([[1, 2, 3, 4], [2, 3, 4, 5], [3, 4, 5, 6], [4, 5, 6, 7]], dtype=float),
]

for A in matrices:
    A_init = A.copy()
    eigenvalues, U = jacobi(A)
    Lambda = np.diag(eigenvalues)

    print(np.round(eigenvalues, 8))
    print(f"{np.linalg.norm(A_init @ U - U @ Lambda):.2e}")
    print()

[-1. -0.  2.]
6.35e-10

[-0.82842712  0.          4.82842712]
1.54e-09

[0. 0. 2. 2.]
0.00e+00

[-1.16515139 -0.          0.         17.16515139]
1.43e-08



In [ ]:
def cholesky_iteration(A, eps=eps, kmax=kmax):

    if not np.allclose(A, A.T):
        raise ValueError("A trebuie sa fie simetrica.")

    # A^(0) = A
    A_prev = A.copy().astype(float)
    A_curr = None
    k = 0

    while k < kmax:
        try:
            # A^(k) = Lk * Lk^T  (factorizare Cholesky)
            L = np.linalg.cholesky(A_prev)
        except np.linalg.LinAlgError:
            raise np.linalg.LinAlgError(f"A^({k}) nu este pozitiv definita; ")

        # A^(k+1) = Lk^T * Lk
        A_curr = L.T @ L

        if np.linalg.norm(A_curr - A_prev) < eps:
            k += 1
            break

        A_prev = A_curr
        k += 1

    return A_curr if A_curr is not None else A_prev, k

In [39]:
np.random.seed(42)
n = 3
M = np.random.rand(n, n)
Ac = M.T @ M + n * np.eye(n)

#matrice diagonala 
#diag(A_final) = valorile proprii ale lui A

A_final, k = cholesky_iteration(Ac)
print(f"({k} iteratii)")
print(f"  A_final =\n{np.round(A_final, 8)}")
print(f"  diag(A_final) = {np.round(np.diag(A_final),8)}")


(337 iteratii)
  A_final =
[[5.77798277 0.         0.        ]
 [0.         3.32337432 0.        ]
 [0.         0.         3.0006382 ]]
  diag(A_final) = [5.77798277 3.32337432 3.0006382 ]


In [43]:
def svd(A):
    p, n = A.shape
    if p <= n:
        raise ValueError(f" p > n ({p}, {n}).")

    # A = U * S * V^T
    U, S, VT = np.linalg.svd(A, full_matrices=True)
    V = VT.T

    singular_values = S

    # rang(A) = numarul de valori singulare strict pozitive
    rank = int(np.sum(S > eps))
    rank_lib = np.linalg.matrix_rank(A)

    # k2(A) = sigma_max / sigma_min_pozitiva
    cond_number = S[0] / S[S > eps][-1]
    cond_number_lib = np.linalg.cond(A)

    # S^I: 1/sigma_i pe diagonala pentru i <= rang, 0 altfel
    SI = np.zeros((n, p))
    SI[:rank, :rank] = np.diag(1.0 / S[:rank])
    # A^I = V * S^I * U^T
    A_I = V @ SI @ U.T

    # A^J = (A^T * A)^-1 * A^T
    ATA = A.T @ A
    try:
        A_J = np.linalg.inv(ATA) @ A.T
        # ||A^I - A^J||_1
        norm_diff = np.linalg.norm(A_I - A_J, ord=1)
    except np.linalg.LinAlgError:
        raise np.linalg.LinAlgError("A.T @ A e singulara.")

    return (
        singular_values,
        rank,
        rank_lib,
        cond_number,
        cond_number_lib,
        A_I,
        A_J,
        norm_diff,
    )

In [ ]:
B1 = np.array([[1, 2], [3, 4], [5, 6], [7, 8]], dtype=float)

np.random.seed(7)
B2 = np.round(np.random.rand(5, 3) * 10).astype(float)

B3 = np.array(
    [
        [1, 2, 3, 1],
        [2, 4, 1, 2],
        [3, 6, 2, 3],
        [4, 8, 0, 4],
        [5, 10, 1, 5],
        [1, 2, 4, 1],
    ],
    dtype=float,
)

for A in [B1, B2, B3]:
    try:
        (
            singular_values,
            rank,
            rank_lib,
            cond_number,
            cond_number_lib,
            A_I,
            A_J,
            norm_diff,
        ) = svd(A)

        print(np.round(singular_values, 6))
        print(rank, rank_lib)
        print(f"{cond_number:.6f}", f"{cond_number_lib:.6f}")
        print(np.round(A_I, 6))
        print(np.round(A_J, 6))
        print(f"{norm_diff:.2e}")
    except (ValueError, np.linalg.LinAlgError) as e:
        print(e)
    print()

=== p=4, n=2, rang plin ===
  singular_values = [14.269095  0.626828]
  rank            = 2  (numpy: 2)
  cond_number     = 22.763964  (numpy: 22.763964)
  A_I =
[[-1.   -0.5   0.    0.5 ]
 [ 0.85  0.45  0.05 -0.35]]
  A_J =
[[-1.   -0.5   0.    0.5 ]
 [ 0.85  0.45  0.05 -0.35]]
  ||A_I - A_J||_1 = 1.12e-14

=== p=5, n=3, aleatoare ===
  singular_values = [20.22041   5.955497  3.109191]
  rank            = 3  (numpy: 3)
  cond_number     = 6.503431  (numpy: 6.503431)
  A_I =
[[-0.093553  0.117221  0.09371  -0.081498  0.052986]
 [ 0.067238  0.09463  -0.04989  -0.054655 -0.051731]
 [ 0.034425 -0.171156  0.00204   0.199261  0.03929 ]]
  A_J =
[[-0.093553  0.117221  0.09371  -0.081498  0.052986]
 [ 0.067238  0.09463  -0.04989  -0.054655 -0.051731]
 [ 0.034425 -0.171156  0.00204   0.199261  0.03929 ]]
  ||A_I - A_J||_1 = 2.78e-16

=== p=6, n=4, rang deficient ===
  Eroare: SVD: A^T*A este singulara (rang(A) = 2 < 4 = n); formula A_J = (A^T A)^-1 A^T necesita rang plin pe coloane.

